# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [38]:
aviation = pd.read_csv(
    "data/AviationData.csv",
    encoding="cp1252",
    low_memory=False
)

print("Rows and columns:", aviation.shape)
display(aviation.head())
aviation.info()
display(aviation.isna().sum().sort_values(ascending=False).head(15))

Rows and columns: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
dtype: int64

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [11]:
# Make a working copy so the original data stays unchanged.
clean = aviation.copy()

# Convert event dates from text into dates Python can filter.
clean["Event.Date"] = pd.to_datetime(clean["Event.Date"], errors="coerce")

raw_count = len(clean)

# Keep only records explicitly identified as airplanes.
clean = clean.loc[clean["Aircraft.Category"].eq("Airplane")].copy()
airplane_count = len(clean)

print("Raw records:", raw_count)
print("After airplane filter:", airplane_count)

# Keep only records confirmed as professional builds.
clean = clean.loc[clean["Amateur.Built"].eq("No")].copy()
professional_count = len(clean)

print("After professional-build filter:", professional_count)

# Keep events on or after the assignment's 1983 cutoff.
clean = clean.loc[
    clean["Event.Date"].ge(pd.Timestamp("1983-01-01"))
].copy()

print("After date filter:", len(clean))
print("Retained date range:", clean["Event.Date"].min(), "to", clean["Event.Date"].max())

Raw records: 88889
After airplane filter: 27617
After professional-build filter: 24417
After date filter: 21447
Retained date range: 1983-03-18 00:00:00 to 2022-12-26 00:00:00


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [15]:
injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

print("Missing values in each injury column:")
display(clean[injury_columns].isna().sum())

missing_per_row = clean[injury_columns].isna().sum(axis=1)
print("Number of rows by how many injury fields are missing:")
display(missing_per_row.value_counts().sort_index())

injury_audit = clean[["Injury.Severity"] + injury_columns].copy()
injury_audit["Missing injury fields"] = injury_audit[injury_columns].isna().sum(axis=1)

display(pd.crosstab(
    injury_audit["Missing injury fields"],
    injury_audit["Injury.Severity"],
    dropna=False
))

display(injury_audit.loc[
    injury_audit["Missing injury fields"].ge(3),
    ["Injury.Severity"] + injury_columns
].head(15))

# Missing injury counts are unknown, so do not replace them with zero.
# The denominator is total reported occupants, not passengers alone.
clean["Total.Occupants.Reported"] = clean[injury_columns].sum(
    axis=1,
    min_count=len(injury_columns)
)

clean["Fatal.Serious.Injuries.Count"] = clean[
    ["Total.Fatal.Injuries", "Total.Serious.Injuries"]
].sum(axis=1, min_count=2)

clean["Fatal.Serious.Injury.Fraction"] = (
    clean["Fatal.Serious.Injuries.Count"]
    / clean["Total.Occupants.Reported"]
)

# A zero or unknown occupant total cannot give a meaningful fraction.
clean.loc[
    clean["Total.Occupants.Reported"].le(0),
    "Fatal.Serious.Injury.Fraction"
] = np.nan

print("Rows with complete occupant counts:",
      clean["Total.Occupants.Reported"].notna().sum())
print("Rows with usable injury fractions:",
      clean["Fatal.Serious.Injury.Fraction"].notna().sum())

display(clean.loc[
    clean["Fatal.Serious.Injury.Fraction"].notna(),
    ["Total.Occupants.Reported",
     "Fatal.Serious.Injuries.Count",
     "Fatal.Serious.Injury.Fraction"]
].head(10))

Missing values in each injury column:


Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64

Number of rows by how many injury fields are missing:


0    18419
1       30
2      238
3     2713
4       47
Name: count, dtype: int64

Injury.Severity,Fatal,Fatal(1),Fatal(10),Fatal(11),Fatal(113),Fatal(12),Fatal(125),Fatal(13),Fatal(131),Fatal(154),...,Fatal(7),Fatal(8),Fatal(9),Fatal(97),Incident,Minor,Non-Fatal,Serious,Unavailable,NaN
Missing injury fields,,,,,,,,,,,,,,,,,,,,,
0,3603,22,0,0,0,0,1,0,0,0,...,0,0,0,0,9,146,13690,108,0,813
1,0,9,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,17,0,0,0
2,0,33,0,0,0,1,0,2,0,2,...,0,2,0,0,4,0,165,0,0,0
3,0,51,6,0,1,2,0,0,1,0,...,10,3,1,0,73,0,2444,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,29,0,2,0,16,0


,Injury.Severity,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
4149,Incident,NaN,NaN,NaN,588.0
4150,Incident,NaN,NaN,NaN,588.0
5957,Incident,NaN,NaN,NaN,289.0
6760,Incident,NaN,NaN,NaN,100.0
6806,Non-Fatal,NaN,NaN,NaN,1.0
7084,Non-Fatal,NaN,NaN,NaN,1.0
7708,Non-Fatal,NaN,NaN,NaN,1.0
8585,Non-Fatal,NaN,NaN,NaN,4.0
8591,Non-Fatal,NaN,NaN,NaN,157.0
8865,Fatal(1),1.0,NaN,NaN,NaN


Rows with complete occupant counts: 18419
Rows with usable injury fractions: 17562


,Total.Occupants.Reported,Fatal.Serious.Injuries.Count,Fatal.Serious.Injury.Fraction
25965,2.0,2.0,1.000000
28791,2.0,0.0,0.000000
28804,1.0,0.0,0.000000
30062,51.0,36.0,0.705882
34275,5.0,0.0,0.000000
37171,9.0,7.0,0.777778
40009,144.0,4.0,0.027778
40930,175.0,169.0,0.965714
47631,336.0,28.0,0.083333
50420,3.0,3.0,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [17]:
display(clean["Aircraft.damage"].value_counts(dropna=False))

# Normalize labels, then map only categories with a known destroyed status.
damage_clean = clean["Aircraft.damage"].astype("string").str.strip().str.casefold()

destroyed_map = {
    "destroyed": True,
    "substantial": False,
    "minor": False
}

clean["Aircraft.Destroyed"] = damage_clean.map(destroyed_map).astype("boolean")

display(clean["Aircraft.Destroyed"].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64

Aircraft.Destroyed
False    17807
True      2316
<NA>      1324
Name: count, dtype: Int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [21]:
# Standardize model labels while preserving the original Model column.
clean["Model.Clean"] = (
    clean["Model"].astype("string").str.strip().str.upper()
)
clean["Model.Clean"] = clean["Model.Clean"].replace("", pd.NA)

rows_before = len(clean)
clean = clean.dropna(subset=["Model.Clean"]).copy()
print("Rows removed for missing Model:", rows_before - len(clean))

# A model label alone is not unique, so identify plane types by make + model.
clean["Plane.Type"] = clean["Make.Clean"] + " | " + clean["Model.Clean"]

print("Rows remaining:", len(clean))
print("Unique plane types:", clean["Plane.Type"].nunique(dropna=True))

display(clean[["Make.Clean", "Model", "Model.Clean", "Plane.Type"]].head(10))

Rows removed for missing Model: 18
Rows remaining: 21429
Unique plane types: 4355


,Make.Clean,Model,Model.Clean,Plane.Type
4149,LOCKHEED,L-1011,L-1011,LOCKHEED | L-1011
4150,BOEING,747,747,BOEING | 747
4171,PIPER,PA-28-140,PA-28-140,PIPER | PA-28-140
4285,DE HAVILLAND,DHC-6,DHC-6,DE HAVILLAND | DHC-6
5957,DOUGLAS,DC-10-10,DC-10-10,DOUGLAS | DC-10-10
5960,LOCKHEED,"LEARSTAR, L-18-56","LEARSTAR, L-18-56","LOCKHEED | LEARSTAR, L-18-56"
6669,SHORT BROTHERS,SD3-30,SD3-30,SHORT BROTHERS | SD3-30
6760,BOEING,727-200,727-200,BOEING | 727-200
6806,BEECH,C35,C35,BEECH | C35
7084,CESSNA,180K,180K,CESSNA | 180K


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [27]:
clean["Engine.Type"] = (
    clean["Engine.Type"].astype("string").str.strip()
)

unknown_engine = clean["Engine.Type"].str.casefold().isin(["unknown", "unk"])
clean.loc[unknown_engine, "Engine.Type"] = pd.NA

display(clean["Engine.Type"].value_counts(dropna=False))

clean["Weather.Condition"] = (
    clean["Weather.Condition"].astype("string").str.strip()
)

unknown_weather = clean["Weather.Condition"].str.casefold().isin(
    ["unknown", "unk"]
)
clean.loc[unknown_weather, "Weather.Condition"] = pd.NA

display(clean["Weather.Condition"].value_counts(dropna=False))

clean["Purpose.of.flight"] = (
    clean["Purpose.of.flight"].astype("string").str.strip()
)

unknown_purpose = clean["Purpose.of.flight"].str.casefold().eq("unknown")
clean.loc[unknown_purpose.fillna(False), "Purpose.of.flight"] = pd.NA

race_variant = clean["Purpose.of.flight"].str.casefold().eq("air race/show")
clean.loc[race_variant.fillna(False), "Purpose.of.flight"] = "Air Race show"

display(clean["Purpose.of.flight"].value_counts(dropna=False))

clean["Broad.phase.of.flight"] = (
    clean["Broad.phase.of.flight"].astype("string").str.strip()
)

unknown_phase = clean["Broad.phase.of.flight"].str.casefold().eq("unknown")
clean.loc[unknown_phase.fillna(False), "Broad.phase.of.flight"] = pd.NA

display(clean["Broad.phase.of.flight"].value_counts(dropna=False))

Engine.Type
Reciprocating      15014
<NA>                4077
Turbo Prop          1251
Turbo Fan            926
Turbo Jet            133
Geared Turbofan       12
Turbo Shaft           11
Electric               5
Name: count, dtype: Int64

Weather.Condition
VMC     17082
<NA>     3279
IMC      1068
Name: count, dtype: Int64

Purpose.of.flight
Personal                     11713
<NA>                          4045
Instructional                 2753
Aerial Application             933
Business                       498
Positioning                    349
Skydiving                      164
Aerial Observation             161
Other Work Use                 155
Flight Test                    120
Ferry                          103
Executive/corporate            103
Banner Tow                      89
Public Aircraft - Federal       52
Air Race show                   52
Public Aircraft                 39
Glider Tow                      35
Public Aircraft - State         24
Firefighting                    17
Public Aircraft - Local         12
ASHO                             5
Air Drop                         3
PUBS                             3
External Load                    1
Name: count, dtype: Int64

Broad.phase.of.flight
<NA>           18619
Landing         1255
Takeoff          509
Cruise           264
Approach         234
Maneuvering      167
Taxi             111
Go-around         89
Descent           73
Climb             58
Standing          47
Other              3
Name: count, dtype: Int64

In [28]:
zero_engine_mask = clean["Number.of.Engines"].eq(0)

print("Zero engine counts set to missing:", zero_engine_mask.sum())
clean.loc[zero_engine_mask, "Number.of.Engines"] = np.nan

print("Missing engine counts after cleaning:",
      clean["Number.of.Engines"].isna().sum())
display(clean["Number.of.Engines"].value_counts(dropna=False).sort_index())

Zero engine counts set to missing: 9
Missing engine counts after cleaning: 2556


Number.of.Engines
1.0    15734
2.0     3004
3.0       40
4.0       93
6.0        1
8.0        1
NaN     2556
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [34]:
required_for_part2 = [
    "Fatal.Serious.Injury.Fraction",
    "Fatal.Serious.Injuries.Count",
    "Total.Occupants.Reported",
    "Aircraft.Destroyed",
    "Weather.Condition",
    "Number.of.Engines",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

non_null_counts = clean.notna().sum()
keep_by_threshold = set(non_null_counts[non_null_counts > 20000].index)
keep_columns = keep_by_threshold | set(required_for_part2)

columns_to_drop = [
    column for column in clean.columns
    if column not in keep_columns
]

print("Required fields preserved below the 20,000 cutoff:")
print(sorted(set(required_for_part2) - keep_by_threshold))

print("\nOther columns that would be dropped:")
print(columns_to_drop) 

clean = clean.drop(columns=columns_to_drop).copy()

print("Final cleaned data shape:", clean.shape)
print("Columns retained:", len(clean.columns))

Required fields preserved below the 20,000 cutoff:
['Fatal.Serious.Injuries.Count', 'Fatal.Serious.Injury.Fraction', 'Number.of.Engines', 'Total.Fatal.Injuries', 'Total.Minor.Injuries', 'Total.Occupants.Reported', 'Total.Serious.Injuries', 'Weather.Condition']

Other columns that would be dropped:
['Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Engine.Type', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Broad.phase.of.flight', 'Report.Status']
Final cleaned data shape: (21429, 28)
Columns retained: 28


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [35]:
cleaned_path = "data/AviationData_cleaned.csv"
clean.to_csv(cleaned_path, index=False, encoding="utf-8")

print("Saved:", cleaned_path)
print("Shape:", clean.shape)

Saved: data/AviationData_cleaned.csv
Shape: (21429, 28)
